### Tic-Tac-Toe Game Environment

This implementation provides a structured environment for a classic 3x3 Tic-Tac-Toe game, designed to be compatible with human play or AI agents (such as Minimax or Reinforcement Learning).

#### Key Components:
1.  **Board Representation**: The 3x3 grid is flattened into a 1D list of length 9. This simplifies indexing (0-8) and is a common practice in machine learning environments.
2.  **Move Validation**: The `make_move` method ensures that a player (represented by 'X' or 'O') only places a mark in an empty square (' ').
3.  **Win Condition Logic**: The `winner` method efficiently checks for a victory after every move by examining three specific vectors related to the last placed piece:
    * **Row**: Checks the horizontal line containing the move.
    * **Column**: Checks the vertical line containing the move.
    * **Diagonal**: If the move is on an even-indexed square (0, 2, 4, 6, or 8), it checks the main and anti-diagonals.
4.  **Utility Methods**: 
    * `available_moves()`: Returns a list of indices that are still empty.
    * `print_board_nums()`: Provides a reference map for players to understand which integer corresponds to which grid position.

In [1]:
class TicTacToeEnv:
    def __init__(self):
        """Initialize a 3x3 empty board, 'X' starts first"""
        # Represent the board as a list of length 9, indices 0-8 map to a 3x3 grid
        self.board = [' ' for _ in range(9)] 
        self.current_winner = None # Track the winner
        
    def print_board(self):
        """Print the current state of the board to the console"""
        for row in [self.board[i*3:(i+1)*3] for i in range(3)]:
            print('| ' + ' | '.join(row) + ' |')
            
    @staticmethod
    def print_board_nums():
        """Print a reference table showing index numbers for each cell"""
        number_board = [[str(i) for i in range(j*3, (j+1)*3)] for j in range(3)]
        for row in number_board:
            print('| ' + ' | '.join(row) + ' |')

    def available_moves(self):
        """Return a list of indices for all currently empty squares"""
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def empty_squares(self):
        """Check if the board still has empty squares"""
        return ' ' in self.board

    def num_empty_squares(self):
        """Return the count of remaining empty squares"""
        return self.board.count(' ')

    def make_move(self, square, letter):
        """
        Place a piece at the specified square.
        If the square is empty, place the piece and check for a win; return True.
        If the square is occupied, return False.
        """
        if self.board[square] == ' ':
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        """Check if placing a 'letter' at 'square' completes a win condition"""
        # 1. Check Row
        row_ind = square // 3
        row = self.board[row_ind*3 : (row_ind+1)*3]
        if all([spot == letter for spot in row]):
            return True
            
        # 2. Check Column
        col_ind = square % 3
        column = [self.board[col_ind+i*3] for i in range(3)]
        if all([spot == letter for spot in column]):
            return True
            
        # 3. Check Diagonals (only indices 0, 2, 4, 6, 8 are on diagonals)
        if square % 2 == 0:
            # Main Diagonal (top-left to bottom-right)
            diagonal1 = [self.board[0], self.board[4], self.board[8]]
            if all([spot == letter for spot in diagonal1]):
                return True
            # Anti-Diagonal (top-right to bottom-left)
            diagonal2 = [self.board[2], self.board[4], self.board[6]]
            if all([spot == letter for spot in diagonal2]):
                return True
                
        return False

# --- Environment Test ---
print("Tic-Tac-Toe Position Reference:")
TicTacToeEnv.print_board_nums()

print("\n--- Testing Simulated Gameplay ---")
# Instantiate the game environment
game = TicTacToeEnv()

# Simulate a few moves
game.make_move(4, 'X') # X moves to the center
game.make_move(0, 'O') # O moves to the top-left corner
game.make_move(2, 'X') # X moves to the top-right corner

game.print_board()
print(f"Current available moves: {game.available_moves()}")
print(f"Is there a winner? {'Yes, it is ' + game.current_winner if game.current_winner else 'No'}")

Tic-Tac-Toe Position Reference:
| 0 | 1 | 2 |
| 3 | 4 | 5 |
| 6 | 7 | 8 |

--- Testing Simulated Gameplay ---
| O |   | X |
|   | X |   |
|   |   |   |
Current available moves: [1, 3, 5, 6, 7, 8]
Is there a winner? No


### Minimax Algorithm for Tic-Tac-Toe

The **Minimax algorithm** is a recursive decision-making algorithm used in two-player zero-sum games. It aims to find the optimal move for a player by assuming the opponent is also playing optimally.

[Image of Minimax algorithm tree for Tic-Tac-Toe]

#### Key Concepts:
1.  **Game Tree Exploration**: The algorithm explores all possible future states of the board. Since Tic-Tac-Toe has a small state space (maximum $9!$ permutations), we can compute the entire tree.
2.  **Maximizing vs. Minimizing**:
    * **AI Turn (Maximizer)**: Chooses the move that results in the highest possible score.
    * **Opponent Turn (Minimizer)**: Chooses the move that results in the lowest possible score for the AI.
3.  **Recursive Scoring**: 
    * Wins are assigned positive values, losses negative, and draws zero.
    * To encourage efficiency, the score is weighted by the number of empty squares: `1 * (empty_squares + 1)`. This makes the AI prefer a win in 3 moves over a win in 5 moves.
4.  **Backtracking**: After "trying" a move in a simulation, the algorithm must revert the board state (`state.board[move] = ' '`) to explore other branches correctly.

In [2]:
import math
import random

# --- 1. Random Player (As a baseline/testing opponent) ---
class RandomPlayer:
    def __init__(self, letter):
        self.letter = letter

    def get_move(self, game):
        return random.choice(game.available_moves())

# --- 2. Minimax Player (Completely independent, no inheritance) ---
class MinimaxPlayer:
    def __init__(self, letter):
        self.letter = letter
        # Counter to track how many future states are evaluated per decision
        self.states_evaluated = 0 

    def get_move(self, game):
        self.states_evaluated = 0
        if len(game.available_moves()) == 9:
            # For the first move, choose randomly to save significant initial computation
            return random.choice(game.available_moves())
        else:
            best_move = self.minimax(game, self.letter)
            print(f"[{self.letter}'s Brain - Minimax] Evaluated {self.states_evaluated} future states for this move.")
            return best_move['position']

    def minimax(self, state, player):
        self.states_evaluated += 1
        max_player = self.letter
        other_player = 'O' if player == 'X' else 'X'

        # --- Terminal Conditions: Check for winner or draw ---
        if state.current_winner == other_player:
            # Weighting score by (empty_squares + 1) encourages faster wins/slower losses
            return {
                'position': None, 
                'score': 1 * (state.num_empty_squares() + 1) if other_player == max_player else -1 * (state.num_empty_squares() + 1)
            }
        elif not state.empty_squares():
            return {'position': None, 'score': 0}

        # --- Recursive Minimax Logic ---
        if player == max_player:
            # AI's turn: Seek to maximize the score
            best = {'position': None, 'score': -math.inf}
            for possible_move in state.available_moves():
                # 1. Try a move
                state.make_move(possible_move, player)
                # 2. Recursively simulate the outcome of this move
                sim_score = self.minimax(state, other_player)
                
                # 3. Backtrack: Undo the move to restore board state for the next iteration
                state.board[possible_move] = ' '
                state.current_winner = None
                sim_score['position'] = possible_move
                
                # 4. Update the best (highest) score
                if sim_score['score'] > best['score']:
                    best = sim_score
            return best
        else:
            # Opponent's turn: They will try to minimize our score
            best = {'position': None, 'score': math.inf}
            for possible_move in state.available_moves():
                # 1. Try a move
                state.make_move(possible_move, player)
                # 2. Recursively simulate the outcome
                sim_score = self.minimax(state, other_player)
                
                # 3. Backtrack
                state.board[possible_move] = ' '
                state.current_winner = None
                sim_score['position'] = possible_move
                
                # 4. Update the best (lowest) score
                if sim_score['score'] < best['score']:
                    best = sim_score
            return best

# --- Game Control Flow ---
def play_game(game, x_player, o_player, print_game=True):
    """Runs a complete Tic-Tac-Toe game and prints each step"""
    if print_game:
        print("--- Game Started ---")
        TicTacToeEnv.print_board_nums()
        print("-" * 20)

    letter = 'X' # X starts by default
    while game.empty_squares():
        if letter == 'O':
            square = o_player.get_move(game)
        else:
            square = x_player.get_move(game)

        if game.make_move(square, letter):
            if print_game:
                print(f"\n{letter} chooses square {square}")
                game.print_board()

            if game.current_winner:
                if print_game:
                    print(f"\nGame Over! {letter} Wins!")
                return letter
            
            letter = 'O' if letter == 'X' else 'X'

    if print_game:
        print("\nGame Over! It's a Tie!")
    return None

# ==========================================
# Run Test
# ==========================================
print("\n=== Match Test: Random Player (X) vs Minimax Player (O) ===")
# Ensure you have already run the block defining TicTacToeEnv
game_env = TicTacToeEnv()
player_x = RandomPlayer('X')
player_o = MinimaxPlayer('O')

play_game(game_env, player_x, player_o, print_game=True)


=== Match Test: Random Player (X) vs Minimax Player (O) ===
--- Game Started ---
| 0 | 1 | 2 |
| 3 | 4 | 5 |
| 6 | 7 | 8 |
--------------------

X chooses square 3
|   |   |   |
| X |   |   |
|   |   |   |
[O's Brain - Minimax] Evaluated 63905 future states for this move.

O chooses square 0
| O |   |   |
| X |   |   |
|   |   |   |

X chooses square 6
| O |   |   |
| X |   |   |
| X |   |   |
[O's Brain - Minimax] Evaluated 1229 future states for this move.

O chooses square 1
| O | O |   |
| X |   |   |
| X |   |   |

X chooses square 4
| O | O |   |
| X | X |   |
| X |   |   |
[O's Brain - Minimax] Evaluated 26 future states for this move.

O chooses square 2
| O | O | O |
| X | X |   |
| X |   |   |

Game Over! O Wins!


'O'

### Alpha-Beta Pruning Optimization

Alpha-Beta pruning is an optimization technique for the Minimax algorithm that stops evaluating a move when at least one possibility has been found that proves the move to be worse than a previously examined move.

[Image of Alpha-Beta Pruning search tree diagram]

#### How it Works:
1.  **Two Variables**:
    * **Alpha ($\alpha$):** The best value that the **Maximizer** currently can guarantee at that level or above. Initialized to $-\infty$.
    * **Beta ($\beta$):** The best value that the **Minimizer** currently can guarantee at that level or above. Initialized to $+\infty$.
2.  **The Pruning Condition**: If at any point $\beta \le \alpha$, it means the current branch cannot possibly affect the final decision. The Maximizer has a better option elsewhere, or the Minimizer has already found a way to force a worse outcome here.
3.  **Efficiency**: While the result is identical to pure Minimax, the computational savings are massive. In the best-case scenario (optimal move ordering), Alpha-Beta can search up to twice as deep as pure Minimax in the same amount of time.

In [3]:
import math
import random

# --- Alpha-Beta Pruning Player (Completely Independent) ---
class AlphaBetaPlayer:
    def __init__(self, letter):
        self.letter = letter
        self.states_evaluated = 0 # State evaluation counter

    def get_move(self, game):
        self.states_evaluated = 0
        if len(game.available_moves()) == 9:
            # First move can be random to save computational power
            return random.choice(game.available_moves())
        else:
            # Initial call: alpha is -inf, beta is +inf
            best_move = self.minimax_alpha_beta(game, self.letter, -math.inf, math.inf)
            print(f"[{self.letter}'s Brain - Alpha-Beta] Evaluated {self.states_evaluated} future states.")
            return best_move['position']

    def minimax_alpha_beta(self, state, player, alpha, beta):
        self.states_evaluated += 1
        max_player = self.letter
        other_player = 'O' if player == 'X' else 'X'

        # --- Terminal Conditions (Same as Minimax) ---
        if state.current_winner == other_player:
            return {
                'position': None, 
                'score': 1 * (state.num_empty_squares() + 1) if other_player == max_player else -1 * (state.num_empty_squares() + 1)
            }
        elif not state.empty_squares():
            return {'position': None, 'score': 0}

        # --- Recursive Logic with Pruning ---
        if player == max_player:
            best = {'position': None, 'score': -math.inf}
            for possible_move in state.available_moves():
                state.make_move(possible_move, player)
                sim_score = self.minimax_alpha_beta(state, other_player, alpha, beta)
                
                # Backtrack
                state.board[possible_move] = ' '
                state.current_winner = None
                sim_score['position'] = possible_move

                # Update best score
                if sim_score['score'] > best['score']:
                    best = sim_score
                
                # ---> Core Pruning Logic <---
                alpha = max(alpha, best['score']) # Max player tries to increase the lower bound
                if beta <= alpha:
                    break # Pruning! The opponent wouldn't let us reach this state
            return best
            
        else:
            best = {'position': None, 'score': math.inf}
            for possible_move in state.available_moves():
                state.make_move(possible_move, player)
                sim_score = self.minimax_alpha_beta(state, other_player, alpha, beta)
                
                # Backtrack
                state.board[possible_move] = ' '
                state.current_winner = None
                sim_score['position'] = possible_move

                # Update best score
                if sim_score['score'] < best['score']:
                    best = sim_score
                
                # ---> Core Pruning Logic <---
                beta = min(beta, best['score']) # Min player tries to decrease the upper bound
                if beta <= alpha:
                    break # Pruning!
            return best

# ==========================================
# Performance Comparison & Live Test
# ==========================================

# 1. Compare computation of pure Minimax vs. Alpha-Beta Pruning
print("=== Efficiency Comparison: Same Scenario, Different Effort ===")
test_game = TicTacToeEnv()
test_game.make_move(4, 'X') # Assume X plays center first
print("Current Board (X in center):")
test_game.print_board()
print("-" * 25)

print("O is calculating its response:")
# Calling pure Minimax from the previous block
pure_mm = MinimaxPlayer('O')
pure_mm.get_move(test_game)

# Calling the new Alpha-Beta player
ab_player = AlphaBetaPlayer('O')
ab_player.get_move(test_game)
print("=" * 50)

# 2. Complete Match Test
print("\n=== Match Test: Random Player (X) vs Alpha-Beta Player (O) ===")
final_game = TicTacToeEnv()
player_x_random = RandomPlayer('X')
player_o_ab = AlphaBetaPlayer('O')

# Using the play_game function from the previous block
play_game(final_game, player_x_random, player_o_ab, print_game=True)

=== Efficiency Comparison: Same Scenario, Different Effort ===
Current Board (X in center):
|   |   |   |
|   | X |   |
|   |   |   |
-------------------------
O is calculating its response:
[O's Brain - Minimax] Evaluated 55505 future states for this move.
[O's Brain - Alpha-Beta] Evaluated 2459 future states.

=== Match Test: Random Player (X) vs Alpha-Beta Player (O) ===
--- Game Started ---
| 0 | 1 | 2 |
| 3 | 4 | 5 |
| 6 | 7 | 8 |
--------------------

X chooses square 1
|   | X |   |
|   |   |   |
|   |   |   |
[O's Brain - Alpha-Beta] Evaluated 3187 future states.

O chooses square 0
| O | X |   |
|   |   |   |
|   |   |   |

X chooses square 2
| O | X | X |
|   |   |   |
|   |   |   |
[O's Brain - Alpha-Beta] Evaluated 268 future states.

O chooses square 3
| O | X | X |
| O |   |   |
|   |   |   |

X chooses square 6
| O | X | X |
| O |   |   |
| X |   |   |
[O's Brain - Alpha-Beta] Evaluated 16 future states.

O chooses square 4
| O | X | X |
| O | O |   |
| X |   |   |

X ch

'O'

### The Ultimate Showdown: Minimax vs. Alpha-Beta Pruning

In this final simulation, we pit two "perfect" players against each other. Since Tic-Tac-Toe is a **solved game**, two optimal players will always result in a draw.

#### Observation Points:
1.  **Logical Parity**: Both players make the best possible moves. Neither can "outsmart" the other because both are exploring the same game tree to its conclusion.
2.  **Computational Divergence**: 
    * **Minimax (Player X)**: Brute-forces the problem, examining every possible branch regardless of whether it is already proven to be a losing path.
    * **Alpha-Beta (Player O)**: Uses mathematical "shortcuts" to ignore branches that cannot possibly improve its outcome.
3.  **Result**: The game ends in a tie, but the output log will show a massive difference in the `states_evaluated` counters. Alpha-Beta achieves "The same wisdom with less effort."

[Image of Minimax vs Alpha-Beta Pruning efficiency graph]

In [4]:
# ==========================================
# ULTIMATE SHOWDOWN: TWO PERFECT AIs
# ==========================================

print("\n" + "="*60)
print(" THE MATCH OF THE CENTURY: PURE MINIMAX (X) VS ALPHA-BETA (O)")
print("="*60 + "\n")

# 1. Instantiate a fresh board environment
championship_game = TicTacToeEnv()

# 2. Instantiate the two top-tier AI players
# X (First Move): Traditional, brute-force Pure Minimax algorithm
player_x_mm = MinimaxPlayer('X')

# O (Second Move): Efficient, optimized Alpha-Beta Pruning algorithm
player_o_ab = AlphaBetaPlayer('O')

# 3. Start the match!
# (Calling the play_game function defined in previous blocks)
play_game(championship_game, player_x_mm, player_o_ab, print_game=True)

print("\n" + "="*60)
print(" MATCH POST-ANALYSIS:")
print(" 1. The result is a Draw, as expected between two perfect players.")
print(" 2. Compare the [Evaluated Future States] for each move:")
print("    Alpha-Beta Pruning maintains perfection while drastically")
print("    reducing computational overhead!")
print("="*60 + "\n")


 THE MATCH OF THE CENTURY: PURE MINIMAX (X) VS ALPHA-BETA (O)

--- Game Started ---
| 0 | 1 | 2 |
| 3 | 4 | 5 |
| 6 | 7 | 8 |
--------------------

X chooses square 0
| X |   |   |
|   |   |   |
|   |   |   |
[O's Brain - Alpha-Beta] Evaluated 2788 future states.

O chooses square 4
| X |   |   |
|   | O |   |
|   |   |   |
[X's Brain - Minimax] Evaluated 7332 future states for this move.

X chooses square 1
| X | X |   |
|   | O |   |
|   |   |   |
[O's Brain - Alpha-Beta] Evaluated 75 future states.

O chooses square 2
| X | X | O |
|   | O |   |
|   |   |   |
[X's Brain - Minimax] Evaluated 198 future states for this move.

X chooses square 6
| X | X | O |
|   | O |   |
| X |   |   |
[O's Brain - Alpha-Beta] Evaluated 17 future states.

O chooses square 3
| X | X | O |
| O | O |   |
| X |   |   |
[X's Brain - Minimax] Evaluated 14 future states for this move.

X chooses square 5
| X | X | O |
| O | O | X |
| X |   |   |
[O's Brain - Alpha-Beta] Evaluated 5 future states.

O chooses